In [106]:
# %pip install torch torchvision -q
# %pip install kornia -q
# %pip install opencv-python -q
# %pip install tqdm matplotlib -q
# %pip install hdbscan -q
# %pip install faiss-cpu -q
# %pip install git+https://github.com/cvg/Hierarchical-Localization.git -q

In [107]:
import os
import gc
import copy
import time
import traceback
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import hdbscan

from pathlib import Path
from tqdm import tqdm
from PIL import Image, ExifTags
from scipy.spatial.transform import Rotation
from collections import defaultdict

from hloc import extract_features, match_features, reconstruction
from hloc.utils.read_write_model import read_model

In [108]:
# Locate data
for _candidate in [
    Path("/Users/sachi/Documents/ml/project/sample_data"),
]:
    if _candidate.exists():
        DATA_DIR = _candidate
        break
else:
    raise FileNotFoundError("Cannot locate competition data directory.")

print(f"DATA_DIR = {DATA_DIR}")

OUTPUT_DIR   = DATA_DIR / "output"
FEATURES_DIR = OUTPUT_DIR / "features"
MATCHES_DIR  = OUTPUT_DIR / "matches"
SFM_DIR      = OUTPUT_DIR / "sfm"
PAIRS_DIR    = OUTPUT_DIR / "pairs"

for d in [FEATURES_DIR, MATCHES_DIR, SFM_DIR, PAIRS_DIR]:
    d.mkdir(exist_ok=True, parents=True)

# Hyperparameters
TOP_K              = 50       # Nearest neighbours per image (was 30)
RESIZE_MAX         = 1600     # local feature resolution (was 1024)
MAX_PAIRS_PER_DS   = 8000     # cap total pairs per dataset
EXHAUSTIVE_THRESH  = 80       # use exhaustive matching if cluster ≤ this size
TIME_BUDGET        = 8.0 * 3600  # 8 hours in seconds (leave 1h margin)

NAN_ROT = "nan;nan;nan;nan;nan;nan;nan;nan;nan"
NAN_T   = "nan;nan;nan"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

START_TIME = time.time()

DATA_DIR = /Users/sachi/Documents/ml/project/sample_data
Device: cpu


In [109]:
print("Loading DINOv2 ViT-S/14 …")
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
dinov2 = dinov2.to(device).eval()
print("DINOv2 ready.")

Loading DINOv2 ViT-S/14 …
DINOv2 ready.


Using cache found in /Users/sachi/.cache/torch/hub/facebookresearch_dinov2_main


In [110]:
# Find all datasets in sample_data (theater only)
datasets = ["imc2023_theater_church"]  # Only theater dataset
print(f"Datasets ({len(datasets)}): {list(datasets)}")

# Verify the dataset exists and count images
for dataset in datasets:
    dataset_dir = find_dataset_dir(DATA_DIR, dataset)
    if dataset_dir:
        image_paths = get_image_paths(dataset_dir)
        print(f"  {dataset}: {len(image_paths)} images")
    else:
        print(f"  {dataset}: directory not found")

Datasets (1): ['imc2023_theater_church']
  imc2023_theater_church: 76 images


In [111]:
# A) Dataset / image discovery
def find_dataset_dir(data_dir: Path, dataset: str):
    # Handle the typo in the actual directory name
    if dataset == "imc2023_theater_church":
        dataset = "imc2023_theather_imc2024_church"
    
    candidates = [
        data_dir / dataset,
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    return None


def get_image_paths(directory: Path) -> list:
    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    return sorted(
        p for p in directory.rglob("*")
        if p.is_file() and p.suffix in exts and "LICENSE" not in p.name
    )


# B) Image pre-processing
def fix_rotation(img: Image.Image) -> Image.Image:
    try:
        exif = img._getexif()
        if exif is None:
            return img
        key = next(k for k, v in ExifTags.TAGS.items() if v == "Orientation")
        o = exif.get(key, 1)
        if   o == 3: img = img.rotate(180, expand=True)
        elif o == 6: img = img.rotate(-90, expand=True)
        elif o == 8: img = img.rotate( 90, expand=True)
    except Exception:
        pass
    return img


# C) Global feature extraction (DINOv2)
def extract_global_features(image_paths: list, dataset_dir: Path):
    feats, names = [], []
    for path in tqdm(image_paths, desc="  DINOv2"):
        try:
            img = Image.open(path).convert("RGB")
            img = fix_rotation(img)
            img = img.resize((224, 224))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.transpose(2, 0, 1)  # HWC → CHW
            with torch.no_grad():
                feat = dinov2(torch.from_numpy(arr).unsqueeze(0).to(device))
            f = feat.cpu().numpy().flatten()
            f = f / (np.linalg.norm(f) + 1e-8)  # L2 normalize
            feats.append(f)
            names.append(str(path.relative_to(dataset_dir)))
        except Exception as e:
            print(f"Warning: skipping {path.name}: {e}")
    if feats:
        global_feats = np.array(feats, dtype=np.float32)
    else:
        global_feats = np.empty((0, 384), dtype=np.float32)
    return names, global_feats


# D) HDBSCAN clustering with noise reassignment
def cluster_images(global_feats: np.ndarray, n_images: int):
    if n_images < 20:
        min_cluster_sz, min_samples = 2, 1
    elif n_images < 60:
        min_cluster_sz, min_samples = 3, 2
    elif n_images < 150:
        min_cluster_sz, min_samples = 5, 3
    else:
        min_cluster_sz, min_samples = 8, 4

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_sz,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(global_feats)
    unique_clusters = sorted(set(labels) - {-1})

    # If no clusters found, put everything in one cluster
    if len(unique_clusters) == 0:
        print("HDBSCAN found 0 clusters -> assigning all to cluster 0")
        return np.zeros(n_images, dtype=int)

    n_noise_before = int((labels == -1).sum())

    # Reassign noise points to nearest cluster center
    if n_noise_before > 0:
        centroids = []
        for c in unique_clusters:
            mask = labels == c
            centroid = global_feats[mask].mean(axis=0)
            centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
            centroids.append(centroid)
        centroids = np.array(centroids, dtype=np.float32)

        noise_idx = np.where(labels == -1)[0]
        noise_feats = global_feats[noise_idx]
        sims = noise_feats @ centroids.T

        for i, idx in enumerate(noise_idx):
            best_pos = np.argmax(sims[i])
            if sims[i, best_pos] > 0.3:
                labels[idx] = unique_clusters[best_pos]

    n_noise_after = int((labels == -1).sum())
    n_clusters = len(set(labels) - {-1})
    print(f"HDBSCAN -> {n_clusters} clusters, "
          f"{n_noise_before} noise -> {n_noise_after} after reassignment")
    return labels


# E) Pair generation (per-cluster, scikit-learn or exhaustive)
def generate_pairs(names, global_feats, labels):
    """Generate pairs: exhaustive for small clusters, scikit-learn top-k + cross-cluster bridges."""
    all_pairs = set()
    unique_clusters = sorted(set(labels) - {-1})

    for c in unique_clusters:
        cluster_idx = np.where(labels == c)[0]
        n_cluster = len(cluster_idx)
        if n_cluster < 2:
            continue

        cluster_names = [names[i] for i in cluster_idx]
        cluster_feats = global_feats[cluster_idx]

        if n_cluster <= EXHAUSTIVE_THRESH:
            for i in range(n_cluster):
                for j in range(i + 1, n_cluster):
                    all_pairs.add((min(cluster_names[i], cluster_names[j]),
                                   max(cluster_names[i], cluster_names[j])))
        else:
            k = min(TOP_K + 1, n_cluster)
            feats_f32 = cluster_feats.astype(np.float32).copy()
            feats_f32 = normalize(feats_f32, norm='l2')
            nbrs = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
            nbrs.fit(feats_f32)
            _, indices = nbrs.kneighbors(feats_f32)
            for i in range(n_cluster):
                for j_pos in range(1, k):
                    j = int(indices[i, j_pos])
                    if 0 <= j < n_cluster and j != i:
                        a, b = cluster_names[i], cluster_names[j]
                        all_pairs.add((min(a, b), max(a, b)))

    # Cross-cluster bridges: top-10 global NN across clusters
    if len(names) > 1 and len(unique_clusters) > 1:
        feats_all = global_feats.astype(np.float32).copy()
        feats_all = normalize(feats_all, norm='l2')
        k_global = min(11, len(names))
        nbrs_all = NearestNeighbors(n_neighbors=k_global, metric='cosine', algorithm='brute')
        nbrs_all.fit(feats_all)
        _, indices_all = nbrs_all.kneighbors(feats_all)

        for i in range(len(names)):
            for j_pos in range(1, k_global):
                j = int(indices_all[i, j_pos])
                if 0 <= j < len(names) and labels[i] != labels[j]:
                    a, b = names[i], names[j]
                    all_pairs.add((min(a, b), max(a, b)))

    pairs = sorted(all_pairs)
    if len(pairs) > MAX_PAIRS_PER_DS:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(pairs), MAX_PAIRS_PER_DS, replace=False)
        pairs = [pairs[i] for i in sorted(idx)]

    return pairs


# F) Extract poses from COLMAP reconstruction
def extract_poses_from_reconstruction(sfm_dir: Path):
    """Read COLMAP model -> dict: image_name -> (rot_str, t_str)."""
    poses = {}
    try:
        cameras, images, points3D = None, None, None

        # Try reading directly from sfm_dir
        for fmt in [".bin", ".txt"]:
            cam_file = sfm_dir / f"cameras{fmt}"
            img_file = sfm_dir / f"images{fmt}"
            if cam_file.exists() and img_file.exists():
                cameras, images, points3D = read_model(str(sfm_dir), ext=fmt)
                break

        # Try numbered subdirectories
        if images is None:
            for sub in sorted(sfm_dir.iterdir()):
                if sub.is_dir():
                    for fmt in [".bin", ".txt"]:
                        cam_file = sub / f"cameras{fmt}"
                        img_file = sub / f"images{fmt}"
                        if cam_file.exists() and img_file.exists():
                            cameras, images, points3D = read_model(str(sub), ext=fmt)
                            break
                    if images is not None:
                        break

        if images is None:
            return poses

        for img_id, img_data in images.items():
            name = img_data.name
            qvec = img_data.qvec   # (qw, qx, qy, qz)
            tvec = img_data.tvec   # (tx, ty, tz)

            # scipy expects (qx, qy, qz, qw)
            R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
            R_mat = R.as_matrix()

            rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
            t_str   = ";".join(f"{v:.10f}" for v in tvec)
            poses[name] = (rot_str, t_str)

    except Exception as e:
        print(f"Warning: error reading model: {e}")
        traceback.print_exc()

    return poses

# G) Extract poses from the pycolmap model object (RELIABLE)
def extract_poses_from_model(model):
    """Extract poses directly from pycolmap.Reconstruction object."""
    poses = {}
    if model is None:
        return poses
    
    try:
        # Get all registered images
        try:
            images = model.images
        except AttributeError:
            return poses
        
        for img_id in images:
            try:
                img = images[img_id]
            except (KeyError, TypeError):
                continue
            
            name = img.name
            R_mat = None
            tvec = None
            
            # Method 1: pycolmap >= 0.6 (cam_from_world)
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec = np.array(cfw.translation)
            except AttributeError:
                pass
            
            # Method 2: pycolmap 0.5 (rotmat + tvec)
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec = np.array(img.tvec)
                except AttributeError:
                    pass
            
            # Method 3: older pycolmap (qvec + tvec)
            if R_mat is None:
                try:
                    qvec = img.qvec
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec = np.array(img.tvec)
                except AttributeError:
                    pass
            
            # Method 4: projection_matrix fallback
            if R_mat is None:
                try:
                    P = np.array(img.projection_matrix())
                    R_mat = P[:3, :3]
                    tvec = P[:3, 3]
                except Exception:
                    pass
            
            if R_mat is None or tvec is None:
                continue
            
            rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
            t_str = ";".join(f"{v:.10f}" for v in tvec)
            poses[name] = (rot_str, t_str)
    
    except Exception as e:
        print(f"Warning: model extraction failed: {e}")
        traceback.print_exc()
    
    return poses

import pycolmap

def _read_single_model_dir(model_dir):
    """Read poses from one COLMAP model directory."""
    poses = {}
    model_dir = Path(model_dir)
    
    # Method 1: pycolmap.Reconstruction (handles all binary formats)
    try:
        rec = pycolmap.Reconstruction(str(model_dir))
        for img_id in rec.images:
            img = rec.images[img_id]
            name = img.name
            R_mat, tvec = None, None
            
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec  = np.array(cfw.translation)
            except Exception:
                pass
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec  = np.array(img.tvec)
                except Exception:
                    pass
            if R_mat is None:
                try:
                    qvec = np.array(img.qvec)
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec  = np.array(img.tvec)
                except Exception:
                    pass
            
            if R_mat is not None and tvec is not None:
                rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                t_str   = ";".join(f"{v:.10f}" for v in tvec)
                poses[name] = (rot_str, t_str)
        if poses:
            return poses
    except Exception:
        pass
    
    # Method 2: hloc read_model fallback
    try:
        for ext in [".bin", ".txt"]:
            if (model_dir / f"cameras{ext}").exists():
                cameras, images, _ = read_model(str(model_dir), ext=ext)
                for img_id, img_data in images.items():
                    qvec = img_data.qvec
                    tvec = img_data.tvec
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                    t_str   = ";".join(f"{v:.10f}" for v in tvec)
                    poses[img_data.name] = (rot_str, t_str)
                if poses:
                    return poses
    except Exception:
        pass
    
    return poses


def read_poses_from_colmap(sfm_dir):
    """Read ALL sub-models from COLMAP reconstruction output."""
    sfm_dir = Path(sfm_dir)
    all_poses = {}
    
    # 1) Read numbered sub-models from models/ directory
    models_dir = sfm_dir / "models"
    if models_dir.exists():
        for sub in sorted(models_dir.iterdir()):
            if sub.is_dir():
                poses = _read_single_model_dir(sub)
                new = 0
                for name, pose in poses.items():
                    if name not in all_poses:
                        all_poses[name] = pose
                        new += 1
                if poses:
                    print(f"    models/{sub.name}: {len(poses)} registered, {new} new")
    
    # 2) Read from main sfm_dir (hloc copies "best" model here)
    main_poses = _read_single_model_dir(sfm_dir)
    new = 0
    for name, pose in main_poses.items():
        if name not in all_poses:
            all_poses[name] = pose
            new += 1
    if main_poses:
        print(f"    main dir: {len(main_poses)} registered, {new} new")
    
    return all_poses

def _read_poses_from_dir(model_dir: Path):
    """Read poses from a single COLMAP model directory."""
    poses = {}
    
    # Method 1: pycolmap.Reconstruction (most reliable)
    try:
        rec = pycolmap.Reconstruction(str(model_dir))
        try:
            reg_ids = rec.reg_image_ids()
        except AttributeError:
            reg_ids = list(rec.images.keys())
        
        for img_id in reg_ids:
            img = rec.images[img_id]
            name = img.name
            R_mat, tvec = None, None
            
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec  = np.array(cfw.translation)
            except AttributeError:
                pass
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec  = np.array(img.tvec)
                except AttributeError:
                    pass
            if R_mat is None:
                try:
                    qvec = np.array(img.qvec)
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec  = np.array(img.tvec)
                except AttributeError:
                    pass
            
            if R_mat is not None and tvec is not None:
                rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                t_str   = ";".join(f"{v:.10f}" for v in tvec)
                poses[name] = (rot_str, t_str)
        
        if poses:
            return poses
    except Exception:
        pass
    
    # Method 2: hloc read_model fallback
    try:
        for ext in [".bin", ".txt"]:
            if (model_dir / f"cameras{ext}").exists():
                cameras, images, _ = read_model(str(model_dir), ext=ext)
                if images:
                    for img_id, img_data in images.items():
                        qvec = img_data.qvec
                        tvec = img_data.tvec
                        R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                        R_mat = R.as_matrix()
                        rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                        t_str   = ";".join(f"{v:.10f}" for v in tvec)
                        poses[img_data.name] = (rot_str, t_str)
                    return poses
    except Exception:
        pass
    
    return poses


def extract_all_poses(sfm_dir: Path):
    """Read ALL sub-models from COLMAP reconstruction."""
    all_poses = {}
    
    # Priority 1: Read each sub-model from models/ directory
    models_dir = sfm_dir / "models"
    if models_dir.exists():
        sub_dirs = sorted(
            [d for d in models_dir.iterdir() if d.is_dir()],
            key=lambda d: int(d.name) if d.name.isdigit() else 0
        )
        for sub in sub_dirs:
            poses = _read_poses_from_dir(sub)
            new = 0
            for name, pose in poses.items():
                if name not in all_poses:
                    all_poses[name] = pose
                    new += 1
            if poses:
                print(f"models/{sub.name}: {len(poses)} registered, {new} new")
    
    # Priority 2: Also read from main sfm_dir (hloc saves "best" model here)
    main_poses = _read_poses_from_dir(sfm_dir)
    new = 0
    for name, pose in main_poses.items():
        if name not in all_poses:
            all_poses[name] = pose
            new += 1
    if main_poses:
        print(f"main dir: {len(main_poses)} registered, {new} new")
    
    return all_poses

In [112]:
import pycolmap
import shutil

# Load sample submission format
sample_submission = pd.read_csv(DATA_DIR / "train_labels.csv")
print(f"Sample submission shape: {sample_submission.shape}")

all_rows = []

for ds_idx, dataset in enumerate(datasets):
    elapsed  = time.time() - START_TIME
    print(f"\n[{ds_idx+1}/{len(datasets)}] Dataset: {dataset}")
    print(f"Time elapsed: {elapsed/60:.1f}min")

    # 1. Locate dataset 
    dataset_dir = find_dataset_dir(DATA_DIR, dataset)
    if dataset_dir is None:
        print("Dataset directory not found")
        continue
    print(f"Path: {dataset_dir}")

    # 2. Collect images 
    image_paths = get_image_paths(dataset_dir)
    n_images = len(image_paths)
    if n_images < 2:
        print(f"Only {n_images} image(s) — skipping")
        continue
    print(f"Images: {n_images}")

    # 3. Output dirs
    ds_feat_dir  = FEATURES_DIR / dataset
    ds_match_dir = MATCHES_DIR  / dataset
    ds_sfm_dir   = SFM_DIR      / dataset
    pairs_file   = PAIRS_DIR    / f"{dataset}_pairs.txt"
    for d in [ds_feat_dir, ds_match_dir, ds_sfm_dir]:
        d.mkdir(exist_ok=True, parents=True)

    # 4. DINOv2 global features (Fast, always run to get scene_map)
    names, global_feats = extract_global_features(image_paths, dataset_dir)
    if not names:
        print("No global features extracted")
        continue

    # 5. HDBSCAN clustering 
    labels = cluster_images(global_feats, len(names))
    scene_map = {}
    unique_labels = sorted(set(labels))
    for i, name in enumerate(names):
        scene_map[name] = f"scene_{labels[i]}"
    
    # 6. Generate pairs 
    pairs = generate_pairs(names, global_feats, labels)
    if not pairs:
        print("No pairs generated")
        continue
    print(f"Pairs: {len(pairs)}")

    # Write pairs file
    with open(pairs_file, "w") as f:
        for a, b in pairs:
            f.write(f"{a} {b}\n")

    # 7. Extract features (SKIP IF EXISTS)
    feature_conf = copy.deepcopy(extract_features.confs["aliked-n16"])
    feature_conf["preprocessing"]["resize_max"] = RESIZE_MAX
    feature_conf['device'] = device
    
    feature_path = extract_features.main(
        feature_conf, dataset_dir, ds_feat_dir, overwrite=False # Changed to False
    )
    print(f"Features: {feature_path.name}")

    # 8. Match ALL pairs (SKIP IF EXISTS)
    matcher_conf = copy.deepcopy(match_features.confs["aliked+lightglue"])
    matcher_conf['device'] = device
    
    match_output = ds_match_dir / f"{matcher_conf['output']}.h5"
    match_path = match_features.main(
        matcher_conf, pairs_file,
        features=feature_path, matches=match_output, overwrite=False # Changed to False
    )
    print(f"Matches: {match_path.name}")

    # 9. Per-cluster SfM (SKIP IF EXISTS)
    all_poses = {}
    unique_clusters = sorted(set(labels) - {-1})

    for c in unique_clusters:
        cluster_idx = [i for i, l in enumerate(labels) if l == c]
        cluster_names = [names[i] for i in cluster_idx]
        if len(cluster_names) < 2:
            continue

        # Filter pairs to within this cluster only
        cluster_set = set(cluster_names)
        cluster_pairs = [(a, b) for a, b in pairs
                         if a in cluster_set and b in cluster_set]
        if not cluster_pairs:
            continue

        # Write cluster-specific pairs file
        cluster_pairs_file = PAIRS_DIR / f"{dataset}_c{c}.txt"
        with open(cluster_pairs_file, "w") as f:
            for a, b in cluster_pairs:
                f.write(f"{a} {b}\n")

        cluster_sfm = ds_sfm_dir / f"cluster_{c}"
        
        # --- RESUME LOGIC ---
        # Try to read existing poses first
        existing_poses = read_poses_from_colmap(cluster_sfm)
        
        # If we found poses, assume it's done and skip
        if existing_poses:
            print(f"  Cluster {c}: Found existing reconstruction ({len(existing_poses)} poses). Skipping.")
            all_poses.update(existing_poses)
            continue
        
        # If not, clean directory and run SfM
        if cluster_sfm.exists():
            shutil.rmtree(cluster_sfm)
        cluster_sfm.mkdir(parents=True)

        try:
            reconstruction.main(
                cluster_sfm, dataset_dir,
                cluster_pairs_file, feature_path, match_path,
                image_list=cluster_names,
            )
        except TypeError:
            try:
                reconstruction.main(
                    cluster_sfm, dataset_dir,
                    cluster_pairs_file, feature_path, match_path,
                )
            except Exception as e2:
                print(f"  Cluster {c}: SfM failed — {e2}")
                continue
        except Exception as e:
            print(f"  Cluster {c}: SfM failed — {e}")
            continue

        # Read new poses
        cluster_poses = read_poses_from_colmap(cluster_sfm)
        n_new = 0
        for name in cluster_names:
            if name in cluster_poses and name not in all_poses:
                all_poses[name] = cluster_poses[name]
                n_new += 1
        print(f"  Cluster {c}: {n_new}/{len(cluster_names)} registered")

    # 10. Finalize for this dataset
    print(f"Total poses: {len(all_poses)}/{n_images}")

# 11. Write rows 
    # Fix dataset name mismatch: submission uses 'imc2023_theather_imc2024_church', not 'imc2023_theater_church'
    submission_dataset_name = "imc2023_theather_imc2024_church"
    sub_df = sample_submission[sample_submission["dataset"] == submission_dataset_name]
    n_valid = 0
    for _, row in sub_df.iterrows():
        name = row["image"]
        if name in all_poses:
            rot_str, t_str = all_poses[name]
            n_valid += 1
        else:
            rot_str = NAN_ROT
            t_str   = NAN_T

        all_rows.append({
            "dataset":            submission_dataset_name,
            "image":              name,
            "scene":              scene_map.get(name, "scene_0"),
            "rotation_matrix":    rot_str,
            "translation_vector": t_str,
        })

    # Prevent division by zero error
    if len(sub_df) > 0:
        print(f"{n_valid}/{len(sub_df)} registered ({100*n_valid/len(sub_df):.1f}%)")
    else:
        print(f"{n_valid} images registered (submission dataframe empty)")

    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTotal time: {(time.time()-START_TIME)/60:.1f} min")

Sample submission shape: (76, 5)

[1/1] Dataset: imc2023_theater_church
Time elapsed: 0.0min
Path: /Users/sachi/Documents/ml/project/sample_data/imc2023_theather_imc2024_church
Images: 76


  DINOv2: 100%|██████████| 76/76 [00:03<00:00, 22.06it/s]
[2026/03/17 03:33:43 hloc INFO] Extracting local features with configuration:
{'device': 'cpu',
 'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
 'output': 'feats-aliked-n16',
 'preprocessing': {'grayscale': False, 'resize_max': 1600}}
[2026/03/17 03:33:43 hloc INFO] Found 76 images in root /Users/sachi/Documents/ml/project/sample_data/imc2023_theather_imc2024_church.
[2026/03/17 03:33:44 hloc INFO] Skipping the extraction.
[2026/03/17 03:33:44 hloc INFO] Matching local features with configuration:
{'device': 'cpu',
 'model': {'features': 'aliked', 'name': 'lightglue'},
 'output': 'matches-aliked-lightglue'}
[2026/03/17 03:33:44 hloc INFO] Skipping the matching.
E20260317 03:33:44.049970 0x209984840 reconstruction.cc:734] rigs, cameras, frames, images, points3D files do not exist at /Users/sachi/Documents/ml/project/sample_data/output/sfm/imc2023_theater_church/cluster_0/models/0
E20260317 03:33:44.133242 0x209984840 re

HDBSCAN -> 2 clusters, 7 noise -> 1 after reassignment
Pairs: 1544
Features: feats-aliked-n16.h5
Matches: matches-aliked-lightglue.h5
    main dir: 50 registered, 50 new
  Cluster 0: Found existing reconstruction (50 poses). Skipping.
    main dir: 25 registered, 25 new
  Cluster 1: Found existing reconstruction (25 poses). Skipping.
Total poses: 75/76
75/76 registered (98.7%)

Total time: 0.1 min


In [113]:
# Create lookup: (dataset, image_name) → (rotation, translation, scene)
pose_lookup = {}
for row in all_rows:
    key = (row["dataset"], row["image"])
    pose_lookup[key] = (
        row["rotation_matrix"],
        row["translation_vector"],
        row["scene"],
    )


# Start from the ORIGINAL sample_submission to keep correct image_id values
submission = sample_submission.copy()

n_filled = 0
for idx, row in submission.iterrows():
    key = (row["dataset"], row["image"])
    if key in pose_lookup:
        rot, t, scene = pose_lookup[key]
        submission.at[idx, "rotation_matrix"]    = rot
        submission.at[idx, "translation_vector"] = t
        submission.at[idx, "scene"]              = scene
        if "nan" not in rot:
            n_filled += 1
    else:
        submission.at[idx, "rotation_matrix"]    = NAN_ROT
        submission.at[idx, "translation_vector"] = NAN_T
        submission.at[idx, "scene"]              = "scene_0"

print(f"Submission: {len(submission)} rows, {n_filled} with valid poses "
      f"({100*n_filled/len(submission):.1f}%)")

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Submission: 76 rows, 75 with valid poses (98.7%)
Saved submission.csv
